In [1]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

In [2]:
npz = np.load("AHE_Sepsis_Data/reduced_format.npz", allow_pickle=True)

In [3]:
static = npz['static_data']

In [4]:
stay_ids = npz['stay_id']
outcome_data = npz['outcome_data']

In [5]:
static.shape, stay_ids.shape, outcome_data.shape

((7741, 4), (7741,), (7741,))

In [6]:
outcomes = np.array([dat[0][0] for dat in outcome_data])

In [7]:
static[:5]

array([[ 0.5       , -0.83651027,  0.66437936,  0.47404638],
       [ 0.5       , -0.23709139,         nan, -0.49869891],
       [-0.5       ,  0.2424437 ,         nan, -0.82557284],
       [ 0.5       , -0.11720762,  1.1229583 ,  1.34045919],
       [ 0.5       ,  1.44128144,         nan,  0.04083997]])

In [8]:
height_mean = np.nanmean(static[:,2])

In [9]:
height_mean

-0.0034128421464264215

In [10]:
static[:5, [0,1,3]]

array([[ 0.5       , -0.83651027,  0.47404638],
       [ 0.5       , -0.23709139, -0.49869891],
       [-0.5       ,  0.2424437 , -0.82557284],
       [ 0.5       , -0.11720762,  1.34045919],
       [ 0.5       ,  1.44128144,  0.04083997]])

In [11]:
data = np.hstack([np.expand_dims(stay_ids, -1), static, np.expand_dims(outcomes, -1)])
data = pd.DataFrame(data, columns=['stay_id', 'gender', 'weight', 'height', 'age', 'mortality'])
data.head()

,stay_id,gender,weight,height,age,mortality
0,30004823.0,0.5,-0.836510,0.664379,0.474046,1.0
1,30005160.0,0.5,-0.237091,NaN,-0.498699,1.0
2,30010785.0,-0.5,0.242444,NaN,-0.825573,1.0
3,30011071.0,0.5,-0.117208,1.122958,1.340459,1.0
4,30011624.0,0.5,1.441281,NaN,0.040840,1.0


In [13]:
hypo_ids = np.load("AHE_Sepsis_Data/hypo_subcohort.npy", allow_pickle=True)
sep_ids = np.load("AHE_Sepsis_Data/sep_subcohort.npy", allow_pickle=True)

In [14]:
data['class'] = data.stay_id.apply(lambda x: 1 if x in sep_ids else -1)

In [19]:
npz2 = np.load("AHE_Sepsis_Data/reduced_format_overlapCohort.npz", allow_pickle=True)
ostatic = npz2['static_data']
osid = npz2['stay_id']
o_outcome_data=npz2['outcome_data']

In [20]:
o_outcomes = np.array([dat[0][0] for dat in o_outcome_data])

In [23]:
osid.shape, ostatic.shape, o_outcomes.shape

((20,), (20, 4), (20,))

In [24]:
o_data = np.hstack([np.expand_dims(osid, -1), ostatic, np.expand_dims(o_outcomes, -1)])
o_data = pd.DataFrame(o_data, columns=['stay_id', 'gender', 'weight', 'height', 'age', 'mortality'])
o_data.head()

,stay_id,gender,weight,height,age,mortality
0,30128855.0,-0.5,1.972007,-1.926453,-0.746163,1.0
1,30187303.0,-0.5,0.369771,0.178772,-0.464471,1.0
2,30595022.0,-0.5,0.827553,NaN,-0.360141,1.0
3,32295004.0,-0.5,0.293474,-1.124462,-0.725297,1.0
4,32379793.0,0.5,0.293474,0.178772,-0.318409,-1.0


In [25]:
o_data['class'] = 0

In [27]:
comb_data = pd.concat([data, o_data], axis=0)
comb_data.shape

(7761, 7)

In [28]:
complete = comb_data[~comb_data.height.isna()]
mssng_ht = comb_data[comb_data.height.isna()]

In [49]:
mssng_ht.head()

,stay_id,gender,weight,height,age,mortality,class
1,30005160.0,0.5,-0.237091,NaN,-0.498699,1.0,1
2,30010785.0,-0.5,0.242444,NaN,-0.825573,1.0,1
4,30011624.0,0.5,1.441281,NaN,0.040840,1.0,1
7,30020307.0,0.5,-0.956394,NaN,-0.100937,1.0,1
10,30026250.0,-0.5,0.781921,NaN,-0.352984,1.0,1


In [29]:
complete.shape, mssng_ht.shape

((3419, 7), (4342, 7))

In [30]:
complete_vals = complete[['gender', 'weight', 'age']].values
mssng_ht_vals = mssng_ht[['gender', 'weight', 'age']].values

In [31]:
tree = cKDTree(complete_vals)

In [40]:
__, idxs = tree.query(mssng_ht_vals, k=10)

In [41]:
idxs.shape

(4342, 10)

In [42]:
idxs[0]

array([ 603,  118, 3313,   62,  110, 2885, 1426, 2229, 2567, 1949])

In [43]:
complete_vals[idxs[0]], mssng_ht_vals[0]

(array([[ 0.5       , -0.23709139, -0.46325475],
        [ 0.5       , -0.23709139, -0.53808131],
        [ 0.5       , -0.23709139, -0.54595779],
        [ 0.5       , -0.17714951, -0.44356355],
        [ 0.5       , -0.23709139, -0.39630467],
        [ 0.5       , -0.29703328, -0.58927843],
        [ 0.5       , -0.17714951, -0.59321668],
        [ 0.5       , -0.29703328, -0.39236643],
        [ 0.5       , -0.11720762, -0.52626659],
        [ 0.5       , -0.11720762, -0.47113123]]),
 array([ 0.5       , -0.23709139, -0.49869891]))

In [44]:
comb_data[~comb_data.height.isna()].iloc[idxs[0]]

,stay_id,gender,weight,height,age,mortality,class
1395,31764368.0,0.5,-0.237091,0.664379,-0.463255,1.0,-1
280,30371730.0,0.5,-0.237091,0.205800,-0.538081,1.0,1
7529,39684203.0,0.5,-0.237091,-0.252779,-0.545958,1.0,1
153,30224713.0,0.5,-0.177150,0.664379,-0.443564,1.0,1
269,30357145.0,0.5,-0.237091,0.022369,-0.396305,1.0,1
6544,38365479.0,0.5,-0.297033,-0.252779,-0.589278,1.0,1
3254,34069747.0,0.5,-0.177150,0.939527,-0.593217,1.0,1
5069,36423058.0,0.5,-0.297033,1.398106,-0.392366,1.0,1
5825,37389789.0,0.5,-0.117208,0.480948,-0.526267,1.0,1
4454,35651412.0,0.5,-0.117208,1.398106,-0.471131,1.0,1


In [52]:
complete.iloc[idxs[2]].height.mean()

0.4992909464682658

In [46]:
imp_hts = []
for ii in range(mssng_ht_vals.shape[0]):
    imp_hts.append(comb_data[~comb_data.height.isna()].iloc[idxs[ii]].height.mean())

In [47]:
comb_data.loc[comb_data.height.isna(),'height'] = imp_hts

In [50]:
comb_data[comb_data.stay_id.isin(mssng_ht.stay_id.unique())]

,stay_id,gender,weight,height,age,mortality,class
1,30005160.0,0.5,-0.237091,0.526806,-0.498699,1.0,1
2,30010785.0,-0.5,0.242444,-0.991129,-0.825573,1.0,1
4,30011624.0,0.5,1.441281,0.499291,0.040840,1.0,1
7,30020307.0,0.5,-0.956394,0.270001,-0.100937,1.0,1
10,30026250.0,-0.5,0.781921,-0.748044,-0.352984,1.0,1
...,...,...,...,...,...,...,...
6,32556356.0,-0.5,1.132741,-0.830588,-0.639224,1.0,0
10,35477442.0,-0.5,-1.690248,-0.280293,1.356089,-1.0,0
12,35715575.0,-0.5,-0.393199,-0.500411,0.102824,1.0,0
13,35729597.0,-0.5,0.217177,-0.848931,-0.281894,1.0,0


In [53]:
static[:5]

array([[ 0.5       , -0.83651027,  0.66437936,  0.47404638],
       [ 0.5       , -0.23709139,         nan, -0.49869891],
       [-0.5       ,  0.2424437 ,         nan, -0.82557284],
       [ 0.5       , -0.11720762,  1.1229583 ,  1.34045919],
       [ 0.5       ,  1.44128144,         nan,  0.04083997]])

In [55]:
comb_data[comb_data['class']!=0][['gender','weight','height','age']].values[:5]

array([[ 0.5       , -0.83651027,  0.66437936,  0.47404638],
       [ 0.5       , -0.23709139,  0.52680568, -0.49869891],
       [-0.5       ,  0.2424437 , -0.99112897, -0.82557284],
       [ 0.5       , -0.11720762,  1.1229583 ,  1.34045919],
       [ 0.5       ,  1.44128144,  0.49929095,  0.04083997]])

In [59]:
npz_dict = dict(npz)
npz2_dict = dict(npz2)

In [60]:
npz_dict['static_data'] = comb_data[comb_data['class']!=0][['gender','weight','height','age']].values
npz2_dict['static_data'] = comb_data[comb_data['class']==0][['gender','weight','height','age']].values

### Split the larger dataset into train/val/test sets

In [64]:
from sklearn.model_selection import train_test_split

In [65]:
# First, combine the intended label columns to aid multi-class stratification...
data['label'] = data['mortality'].astype(str)+data['class'].astype(str)

In [66]:
train_df, val_df = train_test_split(data, test_size=0.25, stratify=data['label'])

In [67]:
val_df, test_df = train_test_split(val_df, test_size=0.8, stratify=val_df['label'])

In [68]:
train_df.shape, val_df.shape, test_df.shape

((5805, 8), (387, 8), (1549, 8))

In [70]:
train_df['mortality'] = train_df['mortality'].astype(float)
val_df['mortality'] = val_df['mortality'].astype(float)
test_df['mortality'] = test_df['mortality'].astype(float)

In [74]:
train_df['mortality'].unique()

array([ 1., -1.])

In [76]:
sum(train_df['mortality']<0) /train_df.shape[0]

0.12282515073212748

In [77]:
sum(train_df['mortality']<0) /train_df.shape[0], sum(val_df['mortality']<0)/val_df.shape[0], sum(test_df.mortality<0)/test_df.shape[0]

(0.12282515073212748, 0.12403100775193798, 0.12265978050355068)

In [78]:
sum(train_df['class']<0)/train_df.shape[0], sum(val_df['class']<0)/val_df.shape[0], sum(test_df['class']<0)/test_df.shape[0]

(0.20051679586563306, 0.20155038759689922, 0.20077469335054873)

In [80]:
train_df.index.values

array([4905, 7128, 4454, ..., 3960, 5499,  576])

In [81]:
val_df.index.values

array([ 138, 4961, 3518,  757, 3850, 1327, 3156, 3031, 7526,  265, 2268,
        135, 1231, 3025, 3167, 2380, 3406, 2888, 2239, 6465,   98, 2518,
       2749, 6224,  304, 3072, 4487, 2944, 2451, 5872, 2718, 7581, 1341,
       5676, 4302, 5571, 3241, 3855, 3670, 3531, 1734,  443, 1872, 2453,
       1167, 1254, 7101, 4812, 4412, 4860, 7632, 3682, 6479, 7502, 3030,
         16, 3383,  437, 2602, 2270, 6809, 2225, 6446, 2905,  808,  943,
       7591, 6082, 1836, 3744, 3243,  894, 2089, 3190,  450,    7,  904,
       2040, 6244, 3628, 2409, 3112, 1298,  986, 7405,  993, 7237, 2598,
       5669, 2982, 1749, 6052, 4078, 2238, 3988, 4978, 7100, 3538, 3910,
        569, 2178, 7298, 6872, 6996, 2288, 2748, 4962, 4770, 2466, 1006,
        249, 6110,  470, 6957, 7206,  632, 2726, 2090, 7279, 7728, 6270,
       2882, 7362, 1858, 4312, 1125, 3202, 2508, 1458, 1677,  697, 2064,
       7515,  435, 7481, 2588, 3337, 2358, 3484, 6945, 2165, 3873, 5030,
       7330, 7392, 3032, 1755, 5181, 4881, 3502, 48

In [82]:
data.shape[0], train_df.shape[0]+val_df.shape[0]+test_df.shape[0]

(7741, 7741)

In [83]:
train_idxs = train_df.index.values
val_idxs = val_df.index.values
test_idxs = test_df.index.values

In [84]:
npz_dict['train_idxs'] = train_idxs
npz_dict['val_idxs'] = val_idxs
npz_dict['test_idxs'] = test_idxs

In [85]:
npz_dict['static_data'][:5]

array([[ 0.5       , -0.83651027,  0.66437936,  0.47404638],
       [ 0.5       , -0.23709139,  0.52680568, -0.49869891],
       [-0.5       ,  0.2424437 , -0.99112897, -0.82557284],
       [ 0.5       , -0.11720762,  1.1229583 ,  1.34045919],
       [ 0.5       ,  1.44128144,  0.49929095,  0.04083997]])

In [86]:
np.savez("AHE_Sepsis_Data/reduced_format.npz", **npz_dict)
np.savez("AHE_Sepsis_Data/reduced_format_overlapCohort.npz", **npz2_dict)

### Playing with forming a mask for the observed temporal data

In [3]:
npz = np.load("../AHE_Sepsis_Data/reduced_format.npz", allow_pickle=True)

In [4]:
npz_dict = dict(npz)
temporal_data = npz_dict['temporal_data']

In [5]:
temp = temporal_data[0]

In [6]:
type(temp), temp.shape

(numpy.ndarray, (46, 39))

In [9]:
temp[:2]

array([[ 0.        ,         nan,         nan,         nan,         nan,
                nan,  0.34587434,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,  0.84616012, -0.5       ],
       [ 1.        , -0.03081148,  2.38363564,  1.90946523,  2.04592607,
        -0.39701281,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,  0.

In [10]:
npz_dict['temporal_columns']

array(['m:timestep', 'o:heart_rate', 'o:sbp', 'o:dbp', 'o:mbp',
       'o:resp_rate', 'o:temperature', 'o:glucose', 'o:so2', 'o:po2',
       'o:pco2', 'o:fio2', 'o:pao2fio2ratio', 'o:ph', 'o:baseexcess',
       'o:chloride', 'o:calcium', 'o:potassium', 'o:sodium', 'o:lactate',
       'o:hematocrit', 'o:hemoglobin', 'o:platelet', 'o:wbc', 'o:albumin',
       'o:aniongap', 'o:bicarbonate', 'o:pt', 'o:ptt', 'o:gcs', 'o:spo2',
       'o:bun', 'o:creatinine', 'o:inr', 'o:bilirubin', 'o:alt', 'o:ast',
       'o:UO', 'o:ventilation'], dtype='<U15')

In [12]:
import torch

In [25]:
intensity = (~torch.tensor(temp)[:, 1:].isnan()).cumsum(axis=0).repeat_interleave(2,0)
mask = (~torch.tensor(temp)[:,1:].isnan())

In [26]:
mask[[0,2],:]

tensor([[False, False, False, False, False,  True, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False,  True,  True],
        [ True,  True,  True,  True,  True, False, False, False,  True,  True,
          True,  True,  True,  True, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False,  True,
         False, False, False, False, False, False, False,  True]])

In [27]:
mask.shape

torch.Size([46, 38])

In [28]:
temp2 = torch.tensor(temp[:,1:]) * mask

In [29]:
temp2[:2]

tensor([[    nan,     nan,     nan,     nan,     nan,  0.3459,     nan,     nan,
             nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan,
             nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan,
             nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan,
             nan,     nan,     nan,     nan,  0.8462, -0.5000],
        [-0.0308,  2.3836,  1.9095,  2.0459, -0.3970,     nan,     nan,     nan,
             nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan,
             nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan,
             nan,     nan,     nan,     nan,  0.3766,  0.3421,     nan,     nan,
             nan,     nan,     nan,     nan,     nan,  0.5000]],
       dtype=torch.float64)

In [22]:
temp[3,:], mask[4,:]

(array([ 3.        , -0.1085243 ,  1.26292009,  1.35674656,  1.30282578,
        -0.39701281,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
        -0.31090886,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,  0.5       ]),
 tensor([2, 2, 2, 2, 2, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 1, 3]))

In [23]:
temp[:3,:]

array([[ 0.        ,         nan,         nan,         nan,         nan,
                nan,  0.34587434,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,  0.84616012, -0.5       ],
       [ 1.        , -0.03081148,  2.38363564,  1.90946523,  2.04592607,
        -0.39701281,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,  0.

In [24]:
mask[[0,2,4],:]

tensor([[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 2],
        [2, 2, 2, 2, 2, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 1, 3]])

In [81]:
# npz_dict = dict(np.load("../AHE_Sepsis_Data/reduced_format.npz", allow_pickle=True))
npz_dict = dict(np.load("../AHE_Sepsis_Data/reduced_format_overlapCohort.npz", allow_pickle=True))
temporal_data = npz_dict['temporal_data']

In [82]:
intensities = []
masks = []
lengths = []
for i in range(len(temporal_data)):
    temp = temporal_data[i]
    lengths.append(temp.shape[0])
    mask = (~torch.tensor(temp)[:,1:].isnan())
    intensity = mask.cumsum(axis=0).repeat_interleave(2,0) # Doubling up to match with the rectilinear interpolation
    
    masks.append(mask.numpy())
    intensities.append(intensity.numpy())


In [83]:
npz_dict['lengths'] = np.array(lengths, dtype='object')
npz_dict['masks'] = np.array(masks, dtype='object')
npz_dict['intensities'] = np.array(intensities, dtype='object')

In [84]:
# np.savez("../AHE_Sepsis_Data/reduced_format.npz", **npz_dict)
np.savez("../AHE_Sepsis_Data/reduced_format_overlapCohort.npz", **npz_dict)

In [30]:
intensity.shape

torch.Size([92, 38])

In [31]:
temp.shape

(46, 39)

In [50]:
np.max(lengths)

73

In [51]:
temp.shape

(16, 39)

In [52]:
temp = temporal_data[:10]

In [53]:
t_len = lengths[:10]

In [54]:
np.max(t_len)

51

In [56]:
from torch.nn.utils.rnn import pad_sequence

In [58]:
temp = [torch.tensor(t) for t in temp]

In [59]:
temp = pad_sequence(temp, batch_first=True, padding_value=0.)

In [60]:
temp.shape

torch.Size([10, 51, 39])

In [61]:
labels = temp[:,1:,:-1]

In [62]:
labels[0,0,:]

tensor([ 1.0000, -0.0308,  2.3836,  1.9095,  2.0459, -0.3970,     nan,     nan,
            nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan,
            nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan,
            nan,     nan,     nan,     nan,     nan,  0.3766,  0.3421,     nan,
            nan,     nan,     nan,     nan,     nan,     nan],
       dtype=torch.float64)

In [63]:
temp[0,1,:]

tensor([ 1.0000, -0.0308,  2.3836,  1.9095,  2.0459, -0.3970,     nan,     nan,
            nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan,
            nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan,
            nan,     nan,     nan,     nan,     nan,  0.3766,  0.3421,     nan,
            nan,     nan,     nan,     nan,     nan,     nan,  0.5000],
       dtype=torch.float64)

In [64]:
npz.files

['static_data',
 'temporal_data',
 'action_data',
 'outcome_data',
 'static_columns',
 'stay_id',
 'temporal_columns',
 'train_idxs',
 'val_idxs',
 'test_idxs']

In [65]:
actions = npz_dict["action_data"][:10]

In [66]:
actions = [torch.tensor(t) for t in actions]
actions = pad_sequence(actions, batch_first=True, padding_value=0.)

In [71]:
actions[4]

tensor([[ 0.],
        [15.],
        [15.],
        [10.],
        [15.],
        [10.],
        [10.],
        [15.],
        [20.],
        [10.],
        [15.],
        [15.],
        [10.],
        [10.],
        [20.],
        [20.],
        [20.],
        [20.],
        [20.],
        [20.],
        [20.],
        [20.],
        [20.],
        [20.],
        [20.],
        [20.],
        [15.],
        [15.],
        [15.],
        [15.],
        [15.],
        [15.],
        [20.],
        [15.],
        [15.],
        [15.],
        [15.],
        [15.],
        [15.],
        [15.],
        [15.],
        [15.],
        [15.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.],
        [ 0.]], dtype=torch.float64)